# Generating Synthetic Gaze with the Statistical Generator

The {py:class}`StatisticalSampleGenerator <pymovements.synthetic.StatisticalSampleGenerator>`
simulates eye movement velocities from parametrised distributions. It implements the velocity
profiles of the simulator described in section 3 of:

> Fuhl, W., Santini, T., Kuebler, T., Castner, N., Rosenstiel, W., & Kasneci, E. (2018).
> *Eye movement simulation and detector creation to reduce laborious parameter adjustments.*
> arXiv:1804.00970.

You give it a sequence of events, it gives each event a velocity profile and takes samples from
the result at the sampling rate you ask for. The events are therefore the ground truth of the
generated samples.

The generator emits a **velocity magnitude**, not a position: the underlying model has no
directions, so nothing here converts the signal into gaze coordinates. Detectors and transforms
that need positions cannot consume its output.

## What you will learn in this tutorial:

* how to generate samples from a sequence of events
* how to plot the generated velocity profiles
* how to change the hyperparameters of the generator
* how to prepare events that have gaps
* how to reproduce a run from its metadata
* how to generate samples for fixations detected in a real recording

## Preparations

We import `pymovements` as the alias `pm` for convenience, and `numpy` and `polars` to compare
signals at the end of this tutorial.

The generation interface is still **experimental**: names and behaviour may change without that
counting as a breaking change, so pin the pymovements version if you depend on it. Each run warns
about this, which we silence here for readability.

In [ ]:
import warnings

import numpy as np
import polars as pl

import pymovements as pm

warnings.filterwarnings('ignore', category=pm.ExperimentalWarning)

## Generating Samples from Events

The events describe what the eyes do and when. Here we write them by hand: two fixations with a
smooth pursuit between them, and a saccade on either side of the pursuit. An event ends where the
next one starts, as the generator needs a value for every point in time.

In [ ]:
events = pm.Events(
    name=['fixation', 'saccade', 'smooth_pursuit', 'saccade', 'fixation'],
    onsets=[0, 200, 240, 640, 680],
    offsets=[200, 240, 640, 680, 900],
    time_unit='ms',
)

events.frame

The constructor of the generator takes the sampling rate and the hyperparameters, which all have
defaults. The events and the seed are passed to
{py:meth}`generate() <pymovements.synthetic.StatisticalSampleGenerator.generate>`, as they belong
to the run and not to the generator.

In [ ]:
generator = pm.synthetic.StatisticalSampleGenerator(sampling_rate=1000)

gaze = generator.generate(events=events, seed=42)

gaze.samples

The output is a {py:class}`Gaze <pymovements.Gaze>` with one sample per millisecond. It carries
the events it was generated from, so the ground truth stays with the samples:

In [ ]:
gaze.events.frame

## Plotting the Velocity Profiles

{py:func}`tsplot() <pymovements.plotting.tsplot>` draws any numeric column against the `time`
column, and `show_events=True` shades the events. Saccades are the Gamma-shaped bursts, smooth
pursuits rise through a sigmoid onset and then hold, and fixations are a small fluctuation just
above zero.

Note that {py:func}`traceplot() <pymovements.plotting.traceplot>` does *not* apply here: it needs
gaze coordinates, and a velocity magnitude has no direction, so there is no path to trace.

In [ ]:
pm.plotting.tsplot(
    gaze,
    'velocity_magnitude',
    show_events=True,
    title='statistical generator, 1000 Hz',
)

## Changing the Hyperparameters

Each random quantity of the generator is a {py:class}`Range <pymovements.synthetic.Range>`. You can
give it as a `(minimum, maximum)` pair for a uniform range, as a single value, or as a `Range` to
choose another distribution. Durations are in milliseconds, velocities and consistencies in degrees
per second. The consistency sets the strength of the jitter along a profile.

The defaults are illustrations, not measured values. Let's lower the saccades, calm the fixations
and let the pursuit grow over its whole velocity range:

In [ ]:
custom_generator = pm.synthetic.StatisticalSampleGenerator(
    sampling_rate=1000,
    saccade_peak_velocity=pm.synthetic.Range(150, 250, 'normal', std=20),
    fixation_consistency=0.5,
    smooth_pursuit_velocity=(20, 60),
    smooth_pursuit_profile='growing',
)

pm.plotting.tsplot(
    custom_generator.generate(events=events, seed=42),
    'velocity_magnitude',
    show_events=True,
    title='custom hyperparameters',
)

Measurement noise is off by default. `noise_percentage` sets the share of the samples that receive
noise, and `noise_velocity` how much each of them changes:

In [ ]:
noisy_generator = pm.synthetic.StatisticalSampleGenerator(
    sampling_rate=1000,
    noise_percentage=10,
    noise_velocity=(20, 60),
)

pm.plotting.tsplot(
    noisy_generator.generate(events=events, seed=42),
    'velocity_magnitude',
    show_events=True,
    title='10 % of the samples with noise',
)

## Preparing Events with Gaps

Detected fixations usually have gaps, as the saccades between them are not part of the sequence.
The generator does not guess what happened in a gap, and rejects such events:

In [ ]:
fixations = pm.Events(
    name=['fixation'] * 4,
    onsets=[0, 280, 610, 870],
    offsets=[240, 550, 820, 1150],
    time_unit='ms',
)

try:
    generator.generate(events=fixations, seed=42)
except ValueError as error:
    print(error)

{py:func}`bridge_gaps() <pymovements.synthetic.bridge_gaps>` fills each gap with a saccade. It is a
separate step because not every gap is a saccade: a long gap is more likely a blink or data loss.
With `max_gap`, gaps longer than the given number of milliseconds are left untouched.

The inserted saccades have an `onset` and a `duration` but no `offset`. As
{py:func}`tsplot() <pymovements.plotting.tsplot>` currently needs an `offset` to shade an event, we
plot bridged events without `show_events=True`.

In [ ]:
bridged = pm.synthetic.bridge_gaps(fixations)

pm.plotting.tsplot(
    generator.generate(events=bridged, seed=42),
    'velocity_magnitude',
    title='four fixations, gaps bridged with saccades',
)

## Requesting a Different Sampling Rate

The profiles are built at `profile_sampling_rate`, 10000 Hz by default, and each sample is the mean
of the profile up to the next sample. A higher sampling rate raises that default with it. The mean
also smooths the jitter, so a lower sampling rate shows less of it, and a consistency tuned at one
sampling rate does not carry over to another:

In [ ]:
rows = []
for sampling_rate in (250, 500, 1000, 2000):
    resampled = pm.synthetic.StatisticalSampleGenerator(sampling_rate=sampling_rate).generate(
        events=events, seed=42,
    )
    rows.append({'sampling rate [Hz]': sampling_rate, 'samples': resampled.samples.height})

pl.DataFrame(rows)

## Reproducing a Run

Each run records the generator, its hyperparameters and the seed in the metadata of its output,
including when the seed was drawn for you.

In [ ]:
unseeded = generator.generate(events=events)

record = unseeded.metadata['synthetic']

print('generator:', record['generator'])
print('drawn seed:', record['seed'])
print('peak velocity:', record['hyperparameters']['saccade_peak_velocity'])

The generator is registered under its name, so
{py:func}`generate_samples() <pymovements.synthetic.generate_samples>` can rebuild it from the
record. Together with the same events, this reproduces the signal exactly:

In [ ]:
repeated = pm.synthetic.generate_samples(
    record['generator'],
    events=events,
    seed=record['seed'],
    **record['hyperparameters'],
)

print('reproduced:', repeated.samples.equals(unseeded.samples))

## Generating Samples for a Real Recording

The events do not have to be handmade. Detect fixations in a recording and the generator follows
them: real event timing, synthetic samples. We use the {ref}`ToyDataset` and keep a short stretch of
its first file:

In [ ]:
dataset = pm.Dataset('ToyDataset', path='data/ToyDataset')
dataset.download()
dataset.load()

recording = dataset.gaze[0]
recording.pix2deg()
recording.pos2vel()
recording.detect('ivt')

detected = pm.Events(recording.events.frame.head(8))

detected.frame

The gaps between these fixations are the real saccades of the recording, which the detector did
not label. We bridge them and use the sampling rate of the recording:

In [ ]:
recording_events = pm.synthetic.bridge_gaps(detected)
sampling_rate = recording.experiment.sampling_rate

from_recording = pm.synthetic.generate_samples(
    'statistical',
    events=recording_events,
    sampling_rate=sampling_rate,
    seed=42,
)

pm.plotting.tsplot(
    from_recording,
    'velocity_magnitude',
    title='conditioned on fixations detected in ToyDataset',
)

### Comparing against the Recording

Because the generator emits a velocity magnitude, the magnitude of the recording's velocity is
what it can be compared against. The defaults were never fitted to a recording, which shows here.
Let's narrow the ranges towards the recording, with values chosen by hand from its fixation speeds
and saccade peaks, and compare the median and the 95th percentile:

In [ ]:
def summarize(values, signal):
    """Summarize a velocity magnitude in degrees per second."""
    return {
        'signal': signal,
        'median': round(float(np.nanmedian(values)), 1),
        'p95': round(float(np.nanpercentile(values, 95)), 1),
    }


window = recording.samples.filter(
    pl.col('time').is_between(detected.frame['onset'][0], detected.frame['offset'][-1]),
)
recorded_magnitude = np.linalg.norm(np.vstack(window['velocity'].to_numpy()), axis=1)

tuned = pm.synthetic.generate_samples(
    'statistical',
    events=recording_events,
    sampling_rate=sampling_rate,
    fixation_consistency=(4, 11),
    saccade_peak_velocity=(80, 260),
    seed=42,
)

pl.DataFrame([
    summarize(recorded_magnitude, 'recording'),
    summarize(from_recording.samples['velocity_magnitude'], 'synthetic, defaults'),
    summarize(tuned.samples['velocity_magnitude'], 'synthetic, tuned'),
])

With the defaults, fixations come out too quiet and saccades too fast. The narrowed ranges bring
both numbers close to the recording, without fitting anything: this compares two summary values
at this sampling rate, and does not show that the signals match in other respects.

## What you have learned in this tutorial:

* generating samples from events with
  {py:class}`StatisticalSampleGenerator <pymovements.synthetic.StatisticalSampleGenerator>`
* plotting the velocity profiles and their events with
  {py:func}`tsplot() <pymovements.plotting.tsplot>`
* setting hyperparameters as pairs, single values or {py:class}`Range <pymovements.synthetic.Range>`
  objects
* filling the gaps of detected events with
  {py:func}`bridge_gaps() <pymovements.synthetic.bridge_gaps>`
* reproducing a run from its metadata with
  {py:func}`generate_samples() <pymovements.synthetic.generate_samples>`
* generating samples for fixations detected in a real recording